In [1]:
from article_utils.util import read_jsonl
import statistics
import os

def relative_file(filename):
    return os.path.join('..', 'data/doccano/2025.05.01_llm_explanation_quality_v2025.03.25', filename)

def admin_consensus_label(row):
    if 'consensus wrong' in row['admin_flags']:
        for status in ['SUPPORTS', 'REFUTES', "NEI"]:
            if status in row['admin_flags']:
                return status
        return None
    return row['consensus.gold_label_name']

df = read_jsonl(relative_file('processed/gold_labels.jsonl.gz'))
df = df[df.is_superceded == False]

print('reviews without a valid consensus:', df[df['consensus.gold_label_name'].isna()].review_id.unique())
df['admin_consensus_label'] = df.apply(admin_consensus_label, axis=1)

USER_LIST = df.username.unique().tolist()
input_df = read_jsonl(relative_file('inputs/examples.jsonl.gz'))
ft_df = input_df[(input_df.model.str.contains(' FT'))].groupby(['review_id', 'claim.flat', 'evidence.flat']).agg({'predicted_label_name': statistics.mode}).reset_index().rename(columns={'predicted_label_name': 'ft_label'})
df = df.merge(ft_df, how='left')
df['ft_flagged'] = df.ft_label != df.gold_label_name

df[df['consensus.gold_label_name'].isna()][['consensus.counts', 'review_id', 'consensus.users', 'ft_label', 'ft_flagged']].drop_duplicates(['review_id'])


reviews without a valid consensus: []


,consensus.counts,review_id,consensus.users,ft_label,ft_flagged


### Now add retrieval status

In [2]:
k_value = 5

chunks_df = read_jsonl('../../data_builder/builds/civicpairs-2025.03.25/document_chunks.jsonl.gz')
print(chunks_df.columns)
facts_df = read_jsonl('../../data_builder/builds/civicfact-2025.03.25/data.jsonl.gz')
print(facts_df.columns)
facts_df = facts_df.explode('evidence')
facts_df['evidence.start'] = facts_df.evidence.str['meta'].str['start']
facts_df['evidence.end'] = facts_df.evidence.str['meta'].str['end']

# overlap_df = chunks_df
ret_df = facts_df.merge(chunks_df)
ret_df['overlaps'] = ret_df.apply(lambda row: max([row['evidence.end'], row['chunk.end']]) - min([row['evidence.start'], row['chunk.start']]), axis=1)
ret_df.loc[ret_df.overlaps < 0, 'overlaps'] = 0
ret_df = ret_df[ret_df.overlaps > 0]
ret_df = ret_df.groupby(['id', 'display_id', 'civic.id', 'document.pmid', 'claim.flat']).agg({'chunk.id': 'unique'}).reset_index()
ret_df = ret_df.groupby(['document.pmid', 'claim.flat']).agg({'chunk.id': list}).reset_index().rename(columns={'chunk.id': 'relevant_chunks'})
ret_df


Index(['acronyms', 'chunk.end', 'chunk.id', 'chunk.overlap', 'chunk.size',
       'chunk.start', 'chunk.text', 'document.content_source',
       'document.length', 'document.pmid', 'document.total_chunks', 'license',
       'partition', 'passages', 'pubtator', 'selections'],
      dtype='object')
Index(['civic.description', 'civic.evidenceDirection', 'civic.flagged',
       'civic.id', 'civic.status', 'claim', 'claim.flat', 'claim.generated',
       'claim.id', 'claim.source', 'conversation', 'display_id',
       'document.doi', 'document.license', 'document.pmcid', 'document.pmid',
       'document.title', 'evidence', 'evidence.flat', 'evidence.generated',
       'evidence.id', 'evidence.selection_method', 'evidence.source',
       'example', 'flagged', 'generated', 'gold_label_name', 'hypothesis.tag',
       'hypothesis.username', 'id', 'in_table', 'original_example',
       'partition', 'pmid.partition', 'reviews'],
      dtype='object')


,document.pmid,claim.flat,relevant_chunks
0,10080613,molecular profile\tevidence type\tsignificance...,"[[10080613:0-1000, 10080613:900-1900, 10080613..."
1,10088816,molecular profile\tevidence type\tsignificance...,"[[10088816:0-1000, 10088816:900-1900, 10088816..."
2,10088816,molecular profile\tevidence type\tsignificance...,"[[10088816:0-1000, 10088816:900-1900, 10088816..."
3,10088816,molecular profile\tevidence type\tsignificance...,"[[10088816:0-1000, 10088816:900-1900, 10088816..."
4,10224221,molecular profile\tevidence type\tsignificance...,[[10224221:0-712]]
...,...,...,...
8679,9829912,molecular profile\tevidence type\tsignificance...,"[[9829912:0-1000, 9829912:900-1718]]"
8680,9829912,molecular profile\tevidence type\tsignificance...,"[[9829912:0-1000, 9829912:900-1718]]"
8681,9829912,molecular profile\tevidence type\tsignificance...,"[[9829912:0-1000, 9829912:900-1718]]"
8682,9839497,molecular profile\tevidence type\tsignificance...,"[[9839497:0-1000, 9839497:900-1540]]"


In [3]:
def check_retrieval_status(row):
    for chunk_set in row.relevant_chunks:
        if len(chunk_set) == len(set(chunk_set) & set(row.evidence_chunks)):
            return True
    return False

def check_partial_retrieval_status(row):
    for chunk_set in row.relevant_chunks:
        if len(set(chunk_set) & set(row.evidence_chunks)) > 0:
            return True
    return False


result_df = df.merge(ret_df)
result_df['retrieval_ok'] = result_df.apply(check_retrieval_status, axis=1)
result_df['partial_retrieval_ok'] = result_df.apply(check_partial_retrieval_status, axis=1)
print(sorted(result_df.columns.tolist()))
result_df.head()

['_text', 'admin_consensus_label', 'admin_flags', 'annotations', 'assignments', 'civic.evidenceDirection', 'civic.evidenceLevel', 'civic.evidenceRating', 'civic.flagged', 'civic.id', 'civic.status', 'claim.flat', 'claim.source', 'comment', 'comment_count', 'comment_date', 'consensus.counts', 'consensus.gold_label_name', 'consensus.gold_label_support', 'consensus.total_labels', 'consensus.users', 'document.content_source', 'document.content_source_x', 'document.doi', 'document.length', 'document.license', 'document.pmcid', 'document.pmid', 'document.title', 'document.total_chunks', 'evidence.flat', 'evidence_chunks', 'filename', 'ft_flagged', 'ft_label', 'gold_label_name', 'id', 'is_confirmed', 'is_confirmed_by', 'is_deleted', 'is_superceded', 'meta', 'partial_retrieval_ok', 'partition', 'project', 'prompt', 'relevant_chunks', 'retrieval_ok', 'review_id', 'review_round', 'score', 'text', 'upload_name', 'username']


,_text,admin_flags,annotations,assignments,civic.evidenceDirection,civic.evidenceLevel,civic.evidenceRating,civic.flagged,civic.id,civic.status,...,score,text,upload_name,username,admin_consensus_label,ft_label,ft_flagged,relevant_chunks,retrieval_ok,partial_retrieval_ok
0,REFUTES. The authors report that individuals w...,[],"[{'created_at': '2025-07-04T22:48:38.399728Z',...","[{'assignee': 'NilanPatel', 'assignee_id': 12,...",SUPPORTS,A,3.0,0.0,1958,ACCEPTED,...,100.0,SUPPORTS\n\nThis evidence supports the claim t...,subset_selection.doccano_input.jsonl 3,NilanPatel,REFUTES,REFUTES,False,"[[28029918:0-1000, 28029918:900-1900, 28029918...",False,True
1,"REFUTES. The evidence writes that ""selumetinib...",[],"[{'created_at': '2025-06-08T20:28:45.373685Z',...","[{'assignee': 'NilanPatel', 'assignee_id': 12,...",SUPPORTS,B,3.0,0.0,7487,ACCEPTED,...,100.0,REFUTES. this variant was actually associated ...,subset_selection.doccano_input.jsonl 3,NilanPatel,REFUTES,REFUTES,False,"[[31151904:0-1000, 31151904:900-1900, 31151904...",False,True
2,NEI/SUPPORTS. The evidence writes generally ab...,[],"[{'created_at': '2025-06-13T16:40:38.132811Z',...","[{'assignee': 'NilanPatel', 'assignee_id': 12,...",SUPPORTS,D,1.0,0.0,3946,ACCEPTED,...,100.0,label: SUPPORTS\nreasoning: The evidence shows...,subset_selection.doccano_input.jsonl 3,NilanPatel,NEI,SUPPORTS,True,"[[26161928:0-926, 26161928:900-926], [26161928...",True,True
3,SUPPORTS. The evidence reports median PFS to b...,"[disease specificity, NEI, consensus wrong]","[{'created_at': '2025-06-13T16:59:01.749687Z',...","[{'assignee': 'jsaliba10', 'assignee_id': 6, '...",SUPPORTS,A,5.0,0.0,6044,ACCEPTED,...,100.0,SUPPORTS\n\nThis evidence supports the claim t...,subset_selection.doccano_input.jsonl 3,NilanPatel,NEI,NEI,False,"[[27480103:0-1000, 27480103:900-1900, 27480103...",True,True
4,NEI/SUPPORTS. The evidence reports median PFS ...,"[disease specificity, NEI, consensus wrong]","[{'created_at': '2025-06-13T16:59:01.749687Z',...","[{'assignee': 'jsaliba10', 'assignee_id': 6, '...",SUPPORTS,A,5.0,0.0,6044,ACCEPTED,...,100.0,SUPPORTS\n\nThis evidence supports the claim t...,subset_selection.doccano_input.jsonl 3,NilanPatel,NEI,NEI,False,"[[27480103:0-1000, 27480103:900-1900, 27480103...",True,True


## Estimate the Retrieval Error

In [4]:
# exclude any where civic.status == REJECTED for retrieval error calculation and model correctness

temp_df = result_df.copy()

# temp_df = temp_df.merge(ft_labels)
total_reviews = temp_df.review_id.nunique()
print('total number of reviews:', total_reviews)

temp_df = temp_df.copy()
temp_df['retrieval_error'] = temp_df.admin_flags.apply(lambda flags: 'retrieval error' in flags)
temp_df['consensus_ok'] = temp_df.admin_flags.apply(lambda flags: 'consensus ok' in flags)
temp_df['impossible_retrieval'] = temp_df.admin_flags.apply(lambda flags: 'impossible retrieval' in flags)
temp_df['civic_error'] = temp_df.admin_flags.apply(lambda flags: 'civic error' in flags)
temp_df['ft_flagged'] = temp_df.gold_label_name == temp_df.ft_label

n = temp_df[temp_df['consensus.gold_label_name'] == temp_df.gold_label_name].review_id.nunique()
print(f'total number of exact matches between known and consensus labelling (n={total_reviews}) {n} ({round(100*n / total_reviews, 2)}%)')

n = temp_df[temp_df.admin_consensus_label == temp_df.gold_label_name].review_id.nunique()
print(f'total number of exact matches between known and admin consensus labelling (n={total_reviews}) {n} ({round(100*n / total_reviews, 2)}%)')

total_ret_errors = temp_df[temp_df['retrieval_error'] == True].review_id.nunique()
retrievable_review = temp_df[(temp_df.impossible_retrieval != True) & (temp_df.gold_label_name != 'NEI')].review_id.nunique()
print(f'manually reviewed retrieval rate: (n={total_ret_errors}  errors of {retrievable_review} reviews) ({round(100 * (retrievable_review - total_ret_errors) / retrievable_review, 2)}%)')

total_ret_errors = temp_df[(temp_df['consensus_ok'] == True) & (temp_df.gold_label_name != temp_df['consensus.gold_label_name'])].review_id.nunique()
print(f'manually reviewed data errors: (n={total_ret_errors}) ({round((100 * total_ret_errors) / total_reviews, 2)}%)')

total_ret_errors = temp_df[(temp_df['consensus_ok'] == False) & (temp_df.admin_consensus_label != temp_df['consensus.gold_label_name'])].review_id.nunique()
print(f'manually reviewed curator consensus errors: (n={total_ret_errors}) ({round((100 * total_ret_errors) / total_reviews, 2)}%)')

total_reviews = temp_df[temp_df.gold_label_name != 'NEI'].review_id.nunique()
n = temp_df[(temp_df.retrieval_ok) & (temp_df.gold_label_name != 'NEI')].review_id.nunique()
print(f'strict expected rate of retrieval success (n={total_reviews}) {n} ({round(100*n / total_reviews, 2)}%)')

total_reviews = temp_df[temp_df.gold_label_name != 'NEI'].review_id.nunique()
n = temp_df[(temp_df.partial_retrieval_ok) & (temp_df.gold_label_name != 'NEI')].review_id.nunique()
print(f'expected rate of partial retrieval success (n={total_reviews}) {n} ({round(100*n / total_reviews, 2)}%)')

# rate of gold label changes to neutral-bot entries
total = temp_df[temp_df['claim.source'].str.startswith('neutral-bot')].review_id.nunique()
n = temp_df[(temp_df['claim.source'].str.startswith('neutral-bot')) & (temp_df['admin_consensus_label'] != temp_df['gold_label_name'])].review_id.nunique()
print(f'rate of neutral-bot generated claims that change label (n= {n} / {total}) {round(100*n / total, 2)}%')

civic_errors_df = temp_df[(temp_df['claim.source'].str.startswith('civic')) & (temp_df['civic.id'].fillna('') != '')].copy()
total = civic_errors_df.review_id.nunique()
n =civic_errors_df[(civic_errors_df.civic_error == True)].review_id.nunique()
print(f'total rate of civic errors in review set ({n} / {total}) {round(100*n / total, 2)}%')

total = civic_errors_df[(civic_errors_df.ft_label != civic_errors_df.gold_label_name)].review_id.nunique()
n =civic_errors_df[ (civic_errors_df.civic_error == True)& (civic_errors_df.ft_flagged == True)].review_id.nunique()
print(f'total rate of civic errors in review set [flagged] ({n} / {total}) {round(100*n / total, 2)}%')

total = civic_errors_df[(civic_errors_df.ft_label == civic_errors_df.gold_label_name)].review_id.nunique()
n =civic_errors_df[(civic_errors_df.civic_error == True)& (civic_errors_df.ft_flagged == False)].review_id.nunique()
print(f'total rate of civic errors in review set [unflagged] ({n} / {total}) {round(100*n / total, 2)}%')

print('all reviews without a consensus label:', temp_df[temp_df['consensus.gold_label_name'].isna()].review_id.nunique())
print(temp_df[temp_df['consensus.gold_label_name'].isna()].review_id.unique())

print('all reivew_id where the admin label does not agree with user consensus:')
temp_df['claim.flat'] = temp_df['claim.flat'].str.split('\n').str[-1]
print(f'examples that need secondary comment review:', temp_df[temp_df.admin_consensus_label != temp_df['consensus.gold_label_name']].review_id.unique())


total number of reviews: 120
total number of exact matches between known and consensus labelling (n=120) 99 (82.5%)
total number of exact matches between known and admin consensus labelling (n=120) 100 (83.33%)
manually reviewed retrieval rate: (n=5  errors of 75 reviews) (93.33%)
manually reviewed data errors: (n=11) (9.17%)
manually reviewed curator consensus errors: (n=1) (0.83%)
strict expected rate of retrieval success (n=78) 17 (21.79%)
expected rate of partial retrieval success (n=78) 78 (100.0%)
rate of neutral-bot generated claims that change label (n= 3 / 30) 10.0%
total rate of civic errors in review set (7 / 90) 7.78%
total rate of civic errors in review set [flagged] (4 / 14) 28.57%
total rate of civic errors in review set [unflagged] (3 / 76) 3.95%
all reviews without a consensus label: 0
[]
all reivew_id where the admin label does not agree with user consensus:
examples that need secondary comment review: ['0724fd61-6048-4a43-a31e-7f24a6a6ad98']


## Model agreement with Consensus Labels

In [5]:
# explanation quality is much higher in the proprietary LLMs so why is there accuracy on labelling so low? do the curators struggle with the same reviews as the models?
import pandas as pd

def model_correctness(row):
    if row['consensus.gold_label_name'] is None or pd.isnull(row['consensus.gold_label_name']):
        return None
    if row.predicted_label_name == row['consensus.gold_label_name']:
        return 1
    return 0

model_df = input_df.copy().merge(df[[c for c in df.columns if c.startswith('consensus.')] + ['review_id', 'claim.flat', 'evidence.flat'] ].drop_duplicates(['review_id', 'evidence.flat', 'claim.flat']))
model_df['consensus_match'] = model_df['predicted_label_name'] == model_df['consensus.gold_label_name']
model_df['correct'] = model_df.apply(model_correctness, axis=1)
model_df['freq'] = 1

# accuracy vs consensus_label
temp_df = model_df[(model_df.partition == 'dev') & (model_df['civic.status'] != 'REJECTED')].groupby(['model']).agg({'consensus_match': 'sum', 'freq': 'sum'}).reset_index()
temp_df['accuracy'] = temp_df.consensus_match / temp_df.freq
print('\nmodel accuracy wrt consensus label (dev-only)')
print(temp_df.sort_values('accuracy'))

temp_df = model_df[(model_df['civic.status'] != 'REJECTED')].groupby(['model']).agg({'consensus_match': 'sum', 'freq': 'sum'}).reset_index()
temp_df['accuracy'] = temp_df.consensus_match / temp_df.freq
print('\nmodel accuracy wrt consensus label (both-partitions)')
print(temp_df.sort_values('accuracy'))

temp_df = model_df.groupby(['model']).agg({'correct': 'sum', 'freq': 'sum'}).reset_index()
temp_df['accuracy'] = temp_df.correct / temp_df.freq
temp_df.sort_values('accuracy')


model accuracy wrt consensus label (dev-only)
                  model  consensus_match  freq  accuracy
2          Llama3.1-70B                8    45  0.177778
1          Gemini2.5pro               23    45  0.511111
3  Llama3.1-70B FT(old)               25    45  0.555556
0                GPT4.1               27    45  0.600000
4        Llama3.1-8B FT               33    45  0.733333

model accuracy wrt consensus label (both-partitions)
                  model  consensus_match  freq  accuracy
2          Llama3.1-70B               37   116  0.318966
3  Llama3.1-70B FT(old)               78   116  0.672414
1          Gemini2.5pro               79   116  0.681034
0                GPT4.1               81   116  0.698276
4        Llama3.1-8B FT               89   116  0.767241


,model,correct,freq,accuracy
2,Llama3.1-70B,38,121,0.314050
3,Llama3.1-70B FT(old),81,121,0.669421
1,Gemini2.5pro,82,121,0.677686
0,GPT4.1,85,121,0.702479
4,Llama3.1-8B FT,93,121,0.768595
